In [21]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss
import pickle
import os

In [22]:
df = pd.read_parquet('../data/processed/merged.parquet')

# Create a combined text field for embedding
df['combined_text'] = (
    df['product_title'].fillna('') + ' ' +
    df['text'].fillna('')
).str.strip()

documents = df['combined_text'].tolist()

In [25]:
with open("documents.pkl", "wb") as f:
    pickle.dump(documents, f)

In [26]:
with open("documents.pkl", "rb") as f:
    documents = pickle.load(f)

In [27]:
# model = SentenceTransformer('all-MiniLM-L6-v2')
# print("Generating embeddings...")
# embeddings = model.encode(documents, show_progress_bar=True, batch_size=256)
# np.save("embeddings.npy", embeddings)
# print(f"Embeddings shape: {embeddings.shape}")

In [28]:
embeddings = np.load("../data/processed/embeddings.npy")
pd.DataFrame(embeddings).head()

,0,1,2,3,4,5,6,7,8,9,...,374,375,376,377,378,379,380,381,382,383
0,-0.121727,-0.020252,0.046526,-0.006109,0.007712,-0.062259,0.044891,-0.013569,-0.108587,-0.000649,...,0.011397,0.031182,-0.040164,0.030477,0.097631,-0.026799,-0.008776,-0.021868,-0.084309,0.123177
1,-0.072484,-0.023346,0.030851,-0.000932,-0.007666,-0.010350,0.079493,-0.054452,-0.022062,-0.002422,...,-0.091487,-0.035153,-0.040535,0.052082,-0.016936,-0.004101,0.010551,-0.179546,0.048705,0.006902
2,-0.030945,0.024194,0.084823,0.045225,0.000579,0.015655,0.057308,-0.015552,-0.102313,0.034400,...,0.007138,0.005565,0.047139,-0.032926,-0.011407,-0.031499,0.009815,-0.020295,-0.028959,0.012848
3,-0.044892,-0.047445,0.074757,0.020974,0.019112,0.012918,0.099731,0.039083,-0.098874,-0.035329,...,-0.042867,-0.008668,0.051742,-0.034643,0.042769,-0.079854,-0.042173,-0.088845,0.067108,0.013565
4,-0.002503,-0.002389,0.053061,-0.037589,-0.011943,-0.074566,0.016549,-0.081292,-0.136509,0.045902,...,-0.036792,0.025469,0.065281,0.040818,-0.043506,0.050661,0.157443,0.040323,0.057327,-0.019156


In [48]:
dimension = embeddings.shape[1]  # 384 for all-MiniLM-L6-v2
faiss.normalize_L2(embeddings)

#index = faiss.IndexFlatL2(dimension)   # L2 (Euclidean) distance
# or use cosine similarity:
index = faiss.IndexFlatIP(dimension) # Inner product (cosine if normalized)

index.add(embeddings)
print(f"Total vectors indexed: {index.ntotal}")

Total vectors indexed: 701528


In [54]:
embeddings.shape

(701528, 384)

In [49]:
os.makedirs('../data/processed/faiss_index', exist_ok=True)

faiss.write_index(index, '../data/processed/faiss_index/index.faiss')

# Save the dataframe so you can retrieve metadata after search
df.to_parquet('../data/processed/faiss_index/metadata.parquet', index=False)

print("Index and metadata saved.")

Index and metadata saved.


In [50]:
def semantic_search(query, top_k=5):
    query_embedding = model.encode([query]).astype('float32')
    faiss.normalize_L2(query_embedding)  # must normalize query too

    distances, indices = index.search(query_embedding, top_k)
    
    results = df.iloc[indices[0]].copy()
    results['distance'] = distances[0]
    
    return pd.DataFrame(results)

In [52]:
results = semantic_search("Sunsceen", top_k=5)
results

,rating,title,text,verified_purchase,product_title,average_rating,price,description,store,details,combined_text,distance
633286,5.0,AMAZING,Absolutely one of the best facial scrubs I’ve ...,True,Bliss Well Yes! Healthy Glow Multivitamin Scru...,4.4,None,[],Bliss,"{""Brand"": ""Bliss"", ""Item Form"": ""Scrub"", ""Prod...",Bliss Well Yes! Healthy Glow Multivitamin Scru...,0.501193
280790,5.0,Great party favors,Used for Memorial Day party with children! Big...,True,Unves 10Sheets American Flag Temporary Tattoos...,4.8,None,[],Unves,"{""Brand"": ""Unves"", ""Size"": ""10 Count (Pack of ...",Unves 10Sheets American Flag Temporary Tattoos...,0.482767
554044,1.0,Junk,The nail clippers broke into 3 pieces the firs...,False,"Manicure Set Nail Clippers Pedicure Kit,Stainl...",2.8,5.6,[7 PCS Multipurpose Manicure Set - professiona...,5th Lily,"{""Package Dimensions"": ""4.29 x 2.87 x 0.83 inc...","Manicure Set Nail Clippers Pedicure Kit,Stainl...",0.479395
421820,5.0,No more clean up = Everything I could hope for.,I order this product so that I wouldn't have t...,False,"Beard Smart, Beard and Mustache Shaving Bib, B...",3.8,None,[],Beard Smart,"{""Is Discontinued By Manufacturer"": ""No"", ""Pac...","Beard Smart, Beard and Mustache Shaving Bib, B...",0.477993
677568,2.0,"Lies! This is not for long, thick hair.",These shower caps were are way too small. They...,True,"4 Pcs Shower Caps for Women and Girl, Long Thi...",3.9,None,[],Tbestmax,"{""Package Dimensions"": ""9.41 x 8.15 x 1.73 inc...","4 Pcs Shower Caps for Women and Girl, Long Thi...",0.476209
